# CV Masterclass 05: Vision Transformers (ViT)

Welcome to Notebook 05. For a decade, Convolutional Neural Networks (CNN) ruled Computer Vision unchallenged. 

Then, taking inspiration from the massive success of NLP Large Language Models, researchers asked: *"What if we just treat an image exactly like a sentence of words, and feed it into a standard Transformer?"*

---

## 🎓 Socratic Deep Check
Ponder this question before we begin:

> *"CNNs possess 'Inductive Bias'—a hardcoded mathematical assumption that pixels near each other are highly correlated. Vision Transformers have absolutely zero Inductive Bias. Why does this lack of mathematical assumption mean ViTs require 10x more training data than CNNs to achieve the same accuracy, but eventually allows them to drastically outperform CNNs when scaled to extreme datasets?"*

---

## Table of Contents
1. **Self-Attention Math:** The foundation.
2. **The Patching Paradigm:** Overcoming $O(N^2)$ pixel death.
3. **Position Embeddings:** Solving spatial blindness.
4. **The `[CLS]` Token:** Sequence to classification mapping.
5. **Inductive Bias:** Scaling laws natively favoring Transformers.
6. **Swin Transformers:** Hierarchical Vision via shifted windows.
7. **CLIP:** Zero-Shot multimodal contrastive learning.
8. **MAE (Masked Autoencoders):** Breaking the JFT-300M data constraint.
9. **Deep Socratic Synthesis:** RoPE vs 1D parameters.


## 1. Self-Attention: The Full Mathematical Derivation

How does a single vector deep in a sequence gather contextual information from every other vector surrounding it?
Assume a sequence $X$ containing $N$ vectors, each of Dimension $D$.

**1. The Learned Projections**
The network learns three unique weight matrices: $W_Q, W_K, W_V$, each of shape $(D, d_k)$.
- **Query ($Q = X \cdot W_Q$):** What I am looking for.
- **Key ($K = X \cdot W_K$):** What I contain.
- **Value ($V = X \cdot W_V$):** What I will output if you select me.

**2. The Attention Score**
To find how tightly Token 1 relates to Token 5, we compute the Dot Product between Token 1's Query and Token 5's Key!
$\text{Score}(Q, K) = Q \cdot K^T$

**3. The $\sqrt{d_k}$ Scaling Factor (Preventing Gradient Death)**
If embedding dimension $d_k$ is large, the sum of the dot product multiplications grows massively. 
Observe the math of a Softmax function $e^{x_i} / \sum e^{x_j}$ natively mapping values to probabilities (0.0 to 1.0):
- Small inputs: `softmax([1.0, 1.01, 0.99])` $\rightarrow [0.332, 0.336, 0.329]$. Gradients are smooth and healthy!
- Huge inputs: `softmax([100, 101, 99])` $\rightarrow [0.26, 0.73, 0.01]$. The maximum value dominates instantly. The probability becomes basically $1.0$, which means the mathematical curve has flattened out. A flat curve has a slope (gradient) of $\mathbf{0.0}$. The network ceases to learn!
Dividing $Q \cdot K^T$ by $\sqrt{d_k}$ squashes the magnitudes down, keeping them in the healthy gradient zone.

**4. The Final Equation**
$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q \cdot K^T}{\sqrt{d_k}}\right) \cdot V$


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SingleHeadAttention(nn.Module):
    def __init__(self, D, d_k):
        super(SingleHeadAttention, self).__init__()
        self.d_k = d_k
        self.W_Q = nn.Linear(D, d_k, bias=False)
        self.W_K = nn.Linear(D, d_k, bias=False)
        self.W_V = nn.Linear(D, d_k, bias=False)
        
    def forward(self, X):
        Q = self.W_Q(X) 
        K = self.W_K(X) 
        V = self.W_V(X) 
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5) 
        attn_weights = F.softmax(scores, dim=-1) 
        output = torch.matmul(attn_weights, V) 
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, D, H):
        super(MultiHeadAttention, self).__init__()
        assert D % H == 0, "Embedding array must be perfectly divisible by Head Count"
        self.d_k = D // H
        self.H = H
        
        self.W_Q = nn.Linear(D, D, bias=False)
        self.W_K = nn.Linear(D, D, bias=False)
        self.W_V = nn.Linear(D, D, bias=False)
        
        self.W_O = nn.Linear(D, D, bias=False)
        
    def forward(self, X):
        B, N, D = X.shape
        Q = self.W_Q(X).view(B, N, self.H, self.d_k).transpose(1, 2) 
        K = self.W_K(X).view(B, N, self.H, self.d_k).transpose(1, 2)
        V = self.W_V(X).view(B, N, self.H, self.d_k).transpose(1, 2)
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5) 
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V) 
        
        output = output.transpose(1, 2).contiguous().view(B, N, D)
        return self.W_O(output)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_seq = torch.randn(2, 5, 512)
mha = MultiHeadAttention(D=512, H=8)

total_params = sum(p.numel() for p in mha.parameters())
assert total_params == 4 * (512**2), "Parameter extraction mathematics failed!"

out_seq = mha(dummy_seq)
print(f"Original Sequence Block: {dummy_seq.shape}")
print(f"MHA      Sequence Block: {out_seq.shape}")
assert out_seq.shape == dummy_seq.shape, "MHA collapsed dimensional logic natively!"


### ⚠️ Common Pitfalls
*   **Scale Factor Forgetting:** Neglecting the `sqrt(d_k)` scaling evaluates softmax exclusively into near $1.0$ binary classifications, stopping all mathematical gradient propagation instantly in deep networks.


## 2. The Patching Paradigm

A Transformer mathematically expects a 1D sequence of tokens (words). An image natively is a 3D grid $(H, W, C)$. 

If we flatten every single pixel into a 'word', a $224\times224$ image becomes $50,176$ words. Transformer Self-Attention scales as exactly $O(N^2)$. Calculating attention across 50,000 tokens evaluates to $2.5$ Billion operations *per layer*. The GPU instantly hits VRAM Out-Of-Memory boundaries.

**The Linear Patches Solution:** We simply slice the image exclusively into larger $16\times16$ blocks. 
A $224\times224$ image divided into $16\times16$ grids guarantees precisely a $14\times14$ matrix of blocks. 
$14 \times 14 = \mathbf{196 \text{ tokens}}$.

We treat each of these $196$ patches as a "word" in a sentence. Now, $N=196$, scaling perfectly securely!

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=16, embed_dim=768):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x) 
        x = x.flatten(2) 
        x = x.transpose(1, 2) 
        return x

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_image = torch.randn(1, 3, 224, 224)
patcher = PatchEmbedding(patch_size=16, embed_dim=768)

tokens = patcher(dummy_image)
print(f"Original Image Grid: {dummy_image.shape}")
print(f"Sequence  of Tokens: {tokens.shape}")
assert tokens.shape == (1, 196, 768), "Patch alignment array extraction failed!"


### ⚠️ Common Pitfalls
*   **Resolution Intolerance:** A model configured for `16x16` patching on a `224x224` image generates exactly $196$ positional parameters. Passing a $256\times256$ inference image triggers exactly $256$ patches natively. The static 1D sequence lengths instantly crash PyTorch tensor matching logic!


## 3. Position Embeddings

If you explicitly scramble words in an NLP sentence, Convolution networks fail completely because they natively trace geometric $(x,y)$ space.
A pure mathematical Attention sequence has zero idea what 'space' is. If you scramble the $196$ tokens randomly, the dot product computes identically universally. It is "Permutation Invariant".

**The Learned Matrix Fix:** Before passing our inputs, we initiate $196$ explicit learnable tensors. We logically add `Parameter[1]` exactly to `Patch[1]`, and `Parameter[196]` directly to `Patch[196]`. Over several billion evaluations, `Parameter[1]` gradients permanently map identifying traits isolating "Top Left Macro Geometry".

### ⚠️ Common Pitfalls
*   **Fixed Parameters vs Learned:** In NLP Transformers, sine waves define positional bounds dynamically. ViT uses entirely 1D learned matrix sequences! They learn perfectly but fail when extrapolating sequences lengths previously entirely unobserved.


## 4. The `[CLS]` Token

After $196$ tokens pass sequentially through the multi-layer Attention blocks, they drop out fully updated. How do we extract exactly $1$ definitive Classification? Average them? Max compress them?

In ViT, we clone a mechanic perfectly derived from BERT. At the very beginning of the evaluation input sequence, we inject an absolute dummy blank element heavily categorized as the **Classification `[CLS]` Token**. 

The sequence climbs to $197$ length. As the sequence traverses universally via Attention matrix dots securely, this `[CLS]` token interacts evenly against the other internal features gathering semantic weight purely globally.

At the network end, we discard tokens $1-196$, and project purely the $197$th `[CLS]` parameter.

### ⚠️ Common Pitfalls
*   **CLS Tensor Dimensional Drift:** The `[CLS]` token acts securely solely sequentially if evaluated exclusively over exactly identical dimensional sizes structurally.


## 5. Inductive Bias & The Scaling Law Paradigm

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why do ViTs perform structurally worse natively than CNNs tightly evaluated on small datasets (ImageNet 1M), but obliterate CNNs structurally running on massive multi-dataset scales (JFT-300M)?*

**A:** 
A purely structural CNN completely forces mathematical tensors uniquely filtering tight localized $3\times3$ grid constraints globally. The math brutally commands: *"Neighboring pixels exclusively define entity logic!"* This implies an enormous architectural **Inductive Bias**. On tight small isolated datasets, the network survives functionally easily because it doesn't have to evaluate "what is 2D reality" natively.

Transformers simply throw $196$ token limits linearly asserting: *"Patch 1 might correlate perfectly with Patch 196 dynamically. Figure all correlation sequences independently!""*

On an isolated $1M$ array, the Transformer struggles randomly. But on a $300$ Million block systematically, the ViT entirely figures out the 2D dimensional concept intrinsically flawlessly! Without the artificial limit of localized pooling filters systematically, the network effortlessly derives macroscopic corner-to-corner semantic correlations uniquely unattainable functionally in CNN geometries linearly.

### ⚠️ Common Pitfalls
*   **Under-Scaling Transformers:** Dropping an unmodified global standard ViT onto a limited medical scan dataset featuring roughly $5,000$ validation templates natively mechanically results fundamentally uniformly identically into massive overfitting boundaries universally. CNN matrices heavily outperform ViTs here securely!


## 6. The Swin Transformer (Hierarchical Vision)

If ViT is functionally superior, why are detection algorithms (like Mask R-CNN) so hesitant strictly functionally leveraging identically raw ViTs structurally?

Because of ViT's massive $O(N^2)$ quadratic explosion uniformly!
A $1024 \times 1024$ macro resolution image generates mathematically definitively $N = 4,096$ patch arrays independently.
Evaluating $Q \cdot K^T$ structurally costs precisely functionally $4096^2 = \mathbf{16.7 \text{ Million Operations per layer natively!}}$ 

The **Swin Transformer** directly solves quadratic explosions purely mechanically by explicitly slicing spatial logic explicitly heavily back tightly! 
- **LOCAL WINDOWS:** It forcefully dynamically breaks massive images into tiny $7 \times 7$ bounded windows. The Attention mathematics purely functions intrinsically **INSIDE** precisely restricted $7 \times 7$ walls.
$O(N \cdot 7^2) = O(49 N) \rightarrow \text{Linearly scales natively!}$

- **SHIFTED WINDOWS:** Local walls completely break the transformer's greatest global power. To regain long range sequences fundamentally homogeneously natively, Layer 2 mechanically rigidly shifts mathematical constraints securely effectively overlapping $50\%$ boundaries diagonally securely, enabling corner elements structurally sequentially organically bleeding context globally precisely.


In [ ]:
import torch

def window_partition(x, window_size):
    """
    Extracts tightly overlapping sequential block matrix views natively.
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows

def window_reverse(windows, window_size, H, W):
    """
    Structurally reinstates raw window tensors explicitly.
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x

def cyclic_shift(x, shift_size):
    """Shift the feature map so windows overlap different neighbors."""
    return torch.roll(x, shifts=(-shift_size, -shift_size), dims=(1, 2))

def create_attention_mask(H, W, window_size, shift_size):
    """
    Creates the attention mask that prevents attention between
    different shifted regions within the same window.
    Returns a (num_windows, window_size*window_size, window_size*window_size)
    mask with -100 for masked positions and 0 for unmasked.
    """
    img_mask = torch.zeros((1, H, W, 1))
    
    # Create 9 distinct regions using 3-slice combinations
    h_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    w_slices = (slice(0, -window_size), slice(-window_size, -shift_size), slice(-shift_size, None))
    
    cnt = 0
    for h in h_slices:
        for w in w_slices:
            img_mask[:, h, w, :] = cnt
            cnt += 1
            
    # Partition into windows and compute attention mask
    mask_windows = window_partition(img_mask, window_size).squeeze(-1)
    mask_windows = mask_windows.view(-1, window_size * window_size)
    
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
    attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
    return attn_mask

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_feature_map = torch.randn(2, 56, 56, 128)
w_size = 7

isolated_windows = window_partition(dummy_feature_map, w_size)
assert isolated_windows.shape == (128, 7, 7, 128), "Window partition logic inherently collapsed natively!"

reconstructed_map = window_reverse(isolated_windows, w_size, 56, 56)
assert reconstructed_map.shape == dummy_feature_map.shape, "Window integration mechanically destructed!"

# Shift Unshift Invertibility Test
dummy_14 = torch.randn(1, 14, 14, 3)
shifted = cyclic_shift(dummy_14, shift_size=3)
unshifted = cyclic_shift(shifted, shift_size=14-3)

print("Cyclic Shift followed by unshift cleanly restores exact origin array intrinsically!")
assert torch.allclose(dummy_14, unshifted, atol=1e-5), "Shift/Unshift structural operation mathematically broke invertible metrics!"

attn_mask = create_attention_mask(14, 14, 7, 3)
print(f"Generated Attention Mask natively restricts {attn_mask.shape} windows smoothly explicitly!\n")


In [ ]:
import torch.nn as nn

class RelativePositionBias(nn.Module):
    """
    Swin uses learnable relative position bias instead of absolute
    position embeddings. For a window of size M, there are
    (2M-1) * (2M-1) possible relative positions between any two tokens.
    """
    def __init__(self, window_size):
        super().__init__()
        self.window_size = window_size
        
        # Table of relative position biases natively
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size - 1)**2, 1)
        )
        
        # Precompute index into bias table for every pair generically
        coords = torch.arange(window_size)
        grid = torch.stack(torch.meshgrid(coords, coords, indexing='ij'))
        grid_flat = grid.flatten(1)  # (2, M*M)
        
        relative_coords = grid_flat[:, :, None] - grid_flat[:, None, :]
        
        # Shift mathematically exactly structurally to start from purely 0 index bounds
        relative_coords[0] += window_size - 1
        relative_coords[1] += window_size - 1
        relative_coords[0] *= 2 * window_size - 1
        
        self.register_buffer('relative_position_index', relative_coords.sum(0).flatten())
        
    def forward(self):
        bias = self.relative_position_bias_table[self.relative_position_index]
        return bias.view(self.window_size**2, self.window_size**2)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
rel_bias = RelativePositionBias(window_size=7)
generated_bias = rel_bias()
print(f"Evaluated Contextual Bias Table Dimensional Arrays strictly: {generated_bias.shape}")
assert generated_bias.shape == (49, 49), "Relative Parameter Bounds structurally critically failed natively!"


### ⚠️ Common Pitfalls
*   **Divisibility Assumptions:** Spatial parameters heavily demand inputs fully divisible structurally securely by the native `window_size`. Feeding an input of $57 \times 57$ mechanically instantly critically crashes natively.


## 7. CLIP (Contrastive Language-Image Pre-Training)

OpenAI's CLIP perfectly bridged structural language semantic matrices cleanly natively fully fundamentally visually seamlessly!

**Architecture:** Combine explicitly $1$ standard ViT Image Encoder securely and entirely independently $1$ standard Text Transformer sequentially.
Train against explicitly identically $400$ Million organic image/caption pairs directly naturally!

**The Contrastive Mechanism (InfoNCE):**
If you pass explicitly heavily strictly a batch of $N = 32,000$ independently matching pairs...
1. The matrices predict dot-products evaluating perfectly structurally securely independently visually completely universally identical arrays.
2. The exact matching target diagonals exclusively maximize.
3. Every mismatched cell natively flawlessly pushes dot-product values systematically smoothly toward zero.

**Zero-Shot Matrix Logic:**
To classify a picture of a dog organically entirely, functionally embed the Image cleanly. Then embed 1000 purely textual sentences safely seamlessly (`"a photo of a car"`, `"a photo of a dog"`).
Multiply the output similarities neatly dynamically securely explicitly! The highest authentically generated score defines the classification output gracefully natively. 
 CLIP achieved an unprecedented 76.2% zero-shot accuracy on ImageNet with zero ImageNet training data, matching a fully supervised ResNet-50 perfectly.


In [ ]:
import torch

# -----------------------------------------------------
# CONCEPT: Zero-Shot Multi-Modal CLIP Extraction Matrix
# -----------------------------------------------------
def mock_conceptual_clip_inference(image_tensor, textual_classes):
    """
    Evaluates exact OpenAI CLIP API structural seamlessly securely evaluated dynamically.
    """
    try:
        import clip
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model, preprocess = clip.load("ViT-B/32", device=device)
        
        text_tokens = clip.tokenize(textual_classes).to(device)
        
        with torch.no_grad():
            image_features = model.encode_image(image_tensor.to(device))
            text_features = model.encode_text(text_tokens)

            logits_per_image, logits_per_text = model(image_tensor.to(device), text_tokens)
            probs = logits_per_image.softmax(dim=-1).cpu().numpy()
            
        print("CLIP API Successfully Extracted Vectors Natively!")
        return probs

    except ImportError:
        print(">> Note: 'clip' package is missing natively. Simulating safely seamlessly exactly cleanly output scores.")
        mock_sims = torch.randn(1, len(textual_classes)).softmax(dim=-1).numpy()
        return mock_sims

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_clip_features = torch.randn(1, 3, 224, 224) # Normal input geometry naturally
class_prompts = ["a photo of a cat", "a photo of a dog", "a picture of a plane"]

probs = mock_conceptual_clip_inference(dummy_clip_features, class_prompts)
print(f"Probabilities efficiently organically evaluated natively: {probs}")


### ⚠️ Common Pitfalls
*   **Prompt Engineering Naivety:** Passing explicitly exact strings identically shapes outputs. Prompting "A picture of a dog" will literally output different embeddings than "A photo of a dog".


## 8. MAE (Masked Autoencoders)

ViT previously structurally completely demanded Google's proprietary JFT-300M dataset to avoid catastrophic overfitting. 
MAE (Masked Autoencoders) is a self-supervised pre-training strategy that allows ViTs to cleanly train efficiently on standard 1.28M ImageNet.

**The Strategy:**
1. Randomly mask out an extreme 75% of the input image patches optimally cleverly functionally.
2. Only feed the visible 25% patches natively flawlessly cleanly purely securely to the Encoder precisely! 
3. Run a lightweight Decoder natively explicitly securely dynamically logically perfectly to gracefully reconstruct the masked locations effectively.

**Why 75% instead of BERT's 15%?**
Images are highly intrinsically spatially redundant! If you mask only 15% of an image effectively, the network trivially interpolates the missing patches natively explicitly by just copying the adjacent pixel blocks safely perfectly safely. Masking 75% forces the model to actually mathematically learn deep structural semantic representations. 

**Computational Hacking:**
This incredibly intelligently reduces compute memory securely explicitly! By purely processing 25% of tokens in the massive Encoder array, computation costs fall mathematically cleanly natively by a staggering 4x identically.


In [ ]:
def random_masking(x, mask_ratio):
    """
    Intelligently generates uniformly masked sequences for MAE autoencoding natively.
    """
    N, L, D = x.shape
    len_keep = int(L * (1 - mask_ratio))
    
    noise = torch.rand(N, L, device=x.device)
    ids_shuffle = torch.argsort(noise, dim=1)
    ids_restore = torch.argsort(ids_shuffle, dim=1) 
    
    ids_keep = ids_shuffle[:, :len_keep]
    
    x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).expand(-1, -1, D))
    
    return x_masked, ids_restore

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_patches = torch.randn(1, 196, 768)

kept_patches, restore_ids = random_masking(dummy_patches, mask_ratio=0.75)

print(f"Original Input Array Sequence: {dummy_patches.shape}")
print(f"Masked Output Geometry Output: {kept_patches.shape}")

assert kept_patches.shape[1] == 49, "Masking geometry collapsed!"


## DeiT (Data-Efficient Image Transformers)

ViT natively required massive Google JFT-300M datasets; directly training it precisely on standard ImageNet (1.28M) achieved only 72% top-1 mathematically drastically underperforming pure intrinsic CNN baselines.

**DeiT** radically bypassed topological inefficiencies dynamically evaluating explicit Knowledge Distillation mappings. It introduces a structural explicit **Distillation Token** automatically inserted functionally mathematically alongside the explicit `[CLS]` token.

During evaluation training sequences cleanly natively:
- `[CLS]` dynamically processes explicitly standard mathematical Cross-Entropy bounds natively.
- The distinct **Distillation Token** aggressively supervises accurately against a dedicated strong Convolutional Neural Network (CNN) Teacher dynamically imitating highly confident structural probability mappings seamlessly globally!

During Inference evaluations practically cleanly:
Exactly perfectly mechanically averages effectively both token results naturally inherently!

**Impact natively:** DeiT-B elegantly closes mechanical accuracy deficiencies effectively achieving exactly 81.8% cleanly seamlessly matching identically purely standard CNNs definitively structurally before MAE autoencoders even inherently existed!

In [ ]:
import torch.nn as nn

class DeiTForwardPass(nn.Module):
    def __init__(self, embed_dim=768, num_classes=1000):
    	super().__init__()
    	
    	# Native Class parameter generation dynamically seamlessly
    	self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
    	self.distill_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
    	
    	self.head = nn.Linear(embed_dim, num_classes)
    	self.head_distill = nn.Linear(embed_dim, num_classes)
    	
    def forward(self, patch_embeddings):
    	B = patch_embeddings.shape[0]
    	
    	# Expand positional identifiers precisely generating spatial matrix dimensions 
    	cls_tokens = self.cls_token.expand(B, -1, -1)
    	distill_tokens = self.distill_token.expand(B, -1, -1)
    	
    	x = torch.cat((cls_tokens, distill_tokens, patch_embeddings), dim=1)
    	
    	# --- Simulated Native Transformer Blocks Execution ---
    	transformer_out = x 
    	# -----------------------------------------------------
    	
    	cls_out = transformer_out[:, 0]
    	distill_out = transformer_out[:, 1]
    	
    	cls_logits = self.head(cls_out)
    	distill_logits = self.head_distill(distill_out)
    	
    	if self.training:
    		return cls_logits, distill_logits
    	else:
    		# Explicit Average natively efficiently
    		return (cls_logits + distill_logits) / 2.0

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_deit_patches = torch.randn(2, 196, 768)
deit_model = DeiTForwardPass(embed_dim=768, num_classes=1000)

deit_model.eval() # Setting directly natively executing effectively cleanly into mathematical average path
deit_logits = deit_model(dummy_deit_patches)

print(f"DeiT Sequence Path Arrays: {dummy_deit_patches.shape}")
print(f"DeiT Structurally Executed Evaluation Averages logically cleanly explicitly: {deit_logits.shape}")
assert deit_logits.shape == (2, 1000), "DeiT structurally cleanly comprehensively failed boundary dimensions explicitly!"


### ⚠️ Common Pitfalls
*   **Positional Embedding Desync:** If you strip out 75% of patches, you MUST successfully ensure you extract the corresponding positional embeddings BEFORE functionally adding them.


## 9. 🎓 Deep Socratic Synthesis

**Q:** *ViT uses learned 1D position embeddings. There are also fixed sinusoidal 2D position embeddings, Relative Position Bias (used in Swin), and ROPE (Rotary Position Embedding). The original ViT paper showed that a ViT trained with 2D fixed embeddings actually transfers better to new image resolutions than one trained with learned 1D embeddings. Why would learned embeddings fail to generalize to different resolutions, and how does ROPE solve the resolution transfer problem mathematically?*

**Ponder deeply:** 
- A learned 1D embedding physically locks `Parameter 1` to `Pixel Pos 1`. If an image scales from $224 \rightarrow 512$, interpolation must occur. Interpolating randomly learned parameters structurally generates massive geometric noise and hallucinates completely false spatial logic natively. 
- Sinusoidal parameters are deterministic mathematical waves. Interpolating them gracefully perfectly correctly maintains strict sequential geometry elegantly. 
- **RoPE (Rotary Position Embeddings):** RoPE encodes structural relationships by rotating accurately mathematically intuitively vectors within their coordinate boundaries. Since angles scale cleanly without losing relative rotations precisely, you regain long range scaling dynamically smoothly effectively organically natively cleanly!
